### Premier League Results Using API Calls From api.football-data.org/V4 and v3.football.api-sports.io

###

### Current Results For 2025/2026 Season

In [3]:
import requests
import pandas as pd
from datetime import datetime

# =============================================================================
# PANDAS DISPLAY SETTINGS - PREVENT TRUNCATION
# =============================================================================

pd.set_option('display.max_rows', None)          # Show all rows
pd.set_option('display.max_columns', None)       # Show all columns
pd.set_option('display.width', None)             # Auto-detect terminal width
pd.set_option('display.max_colwidth', None)  

# =============================================================================
# CONFIGURATION
# =============================================================================

API_KEY = "your_api_key_here"  # Replace with your API-Football key
BASE_URL = "https://api.football-data.org/v4"

headers = {
    "X-Auth-Token": API_KEY
}

# =============================================================================
# HELPER FUNCTION
# =============================================================================

def make_request(endpoint):
    """Make API request and return JSON data"""
    url = f"{BASE_URL}{endpoint}"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# =============================================================================
# GET PREMIER LEAGUE STANDINGS
# =============================================================================

def get_standings():
    """Extract current Premier League table"""
    data = make_request("/competitions/PL/standings")
    
    if data:
        table = data['standings'][0]['table']
        
        df = pd.DataFrame([{
            'position': team['position'],
            'team': team['team']['name'],
            'played': team['playedGames'],
            'won': team['won'],
            'drawn': team['draw'],
            'lost': team['lost'],
            'goals_for': team['goalsFor'],
            'goals_against': team['goalsAgainst'],
            'goal_difference': team['goalDifference'],
            'points': team['points'],
            'form': team.get('form', 'N/A')
        } for team in table])
        
        return df
    return None

# =============================================================================
# GET FIXTURES/MATCHES
# =============================================================================

def get_matches(status=None, matchday=None, date_from=None, date_to=None):
    """
    Extract Premier League fixtures
    
    Parameters:
        status: SCHEDULED, LIVE, IN_PLAY, PAUSED, FINISHED, POSTPONED, CANCELLED
        matchday: Specific matchday number (1-38)
        date_from: Start date (YYYY-MM-DD)
        date_to: End date (YYYY-MM-DD)
    """
    endpoint = "/competitions/PL/matches"
    params = []
    
    if status:
        params.append(f"status={status}")
    if matchday:
        params.append(f"matchday={matchday}")
    if date_from:
        params.append(f"dateFrom={date_from}")
    if date_to:
        params.append(f"dateTo={date_to}")
    
    if params:
        endpoint += "?" + "&".join(params)
    
    data = make_request(endpoint)
    
    if data:
        matches = data['matches']
        
        df = pd.DataFrame([{
            'matchday': match['matchday'],
            'date': match['utcDate'][:10],
            'time': match['utcDate'][11:16],
            'home_team': match['homeTeam']['name'],
            'away_team': match['awayTeam']['name'],
            'home_score': match['score']['fullTime']['home'],
            'away_score': match['score']['fullTime']['away'],
            'status': match['status'],
            'venue': match.get('venue', 'N/A')
        } for match in matches])
        
        return df
    return None

# =============================================================================
# GET TOP SCORERS
# =============================================================================

def get_top_scorers(limit=20):
    """Extract Premier League top scorers"""
    data = make_request(f"/competitions/PL/scorers?limit={limit}")
    
    if data:
        scorers = data['scorers']
        
        df = pd.DataFrame([{
            'player': scorer['player']['name'],
            'team': scorer['team']['name'],
            'goals': scorer['goals'],
            'assists': scorer.get('assists', 0),
            'penalties': scorer.get('penalties', 0),
            'matches_played': scorer['playedMatches']
        } for scorer in scorers])
        
        df['goals_per_game'] = round(df['goals'] / df['matches_played'], 2)
        
        return df
    return None

# =============================================================================
# GET ALL TEAMS
# =============================================================================

def get_teams():
    """Extract all Premier League teams with details"""
    data = make_request("/competitions/PL/teams")
    
    if data:
        teams = data['teams']
        
        df = pd.DataFrame([{
            'name': team['name'],
            'short_name': team['shortName'],
            'tla': team['tla'],
            'founded': team.get('founded', 'N/A'),
            'stadium': team.get('venue', 'N/A'),
            'website': team.get('website', 'N/A')
        } for team in teams])
        
        return df
    return None

# =============================================================================
# EXAMPLE USAGE
# =============================================================================

if __name__ == "__main__":
    
    print("=" * 60)
    print("PREMIER LEAGUE STANDINGS")
    print("=" * 60)
    standings = get_standings()
    if standings is not None:
        print(standings.to_string(index=False))
    
    print("\n")
    
    print("=" * 60)
    print("TOP SCORERS")
    print("=" * 60)
    scorers = get_top_scorers(limit=10)
    if scorers is not None:
        print(scorers.to_string(index=False))
    
    print("\n")
    
    print("=" * 60)
    print("UPCOMING FIXTURES")
    print("=" * 60)
    upcoming = get_matches(status="SCHEDULED")
    if upcoming is not None:
        print(upcoming.head(10).to_string(index=False))
    
    print("\n")
    
    print("=" * 60)
    print("TEAMS")
    print("=" * 60)
    teams = get_teams()
    if teams is not None:
        print(teams.to_string(index=False))


PREMIER LEAGUE STANDINGS
 position                       team  played  won  drawn  lost  goals_for  goals_against  goal_difference  points form
        1                 Arsenal FC      20   15      3     2         40             14               26      48 None
        2         Manchester City FC      20   13      3     4         44             18               26      42 None
        3             Aston Villa FC      20   13      3     4         33             24                9      42 None
        4               Liverpool FC      20   10      4     6         32             28                4      34 None
        5                 Chelsea FC      20    8      7     5         33             22               11      31 None
        6       Manchester United FC      20    8      7     5         34             30                4      31 None
        7               Brentford FC      20    9      3     8         32             28                4      30 None
        8             S

###

### Inital Attempt of API Call For 2024/2025 Season

###

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# =============================================================================
# PANDAS DISPLAY SETTINGS
# =============================================================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# =============================================================================
# CONFIGURATION
# =============================================================================

API_KEY = "your_api_key_here"  # Replace with your API-Football key
BASE_URL = "https://v3.football.api-sports.io"

# Premier League ID = 39, Season = 2024
LEAGUE_ID = 39
SEASON = 2024

headers = {
    "x-apisports-key": API_KEY
}

# =============================================================================
# HELPER FUNCTION
# =============================================================================

def make_request(endpoint, params=None):
    """Make API request and return JSON data"""
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        # Check for API errors
        if data.get('errors'):
            print(f"API Error: {data['errors']}")
            return None
        return data
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# =============================================================================
# GET LEAGUE STANDINGS
# =============================================================================

def get_standings():
    """Get current Premier League table"""
    data = make_request("standings", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        standings = data['response'][0]['league']['standings'][0]
        
        df = pd.DataFrame([{
            'position': team['rank'],
            'team': team['team']['name'],
            'played': team['all']['played'],
            'won': team['all']['win'],
            'drawn': team['all']['draw'],
            'lost': team['all']['lose'],
            'goals_for': team['all']['goals']['for'],
            'goals_against': team['all']['goals']['against'],
            'goal_difference': team['goalsDiff'],
            'points': team['points'],
            'form': team['form']
        } for team in standings])
        
        return df
    return None

# =============================================================================
# GET TOP SCORERS
# =============================================================================

def get_top_scorers(limit=20):
    """Get Premier League top scorers"""
    data = make_request("players/topscorers", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        scorers = data['response'][:limit]
        
        df = pd.DataFrame([{
            'player': scorer['player']['name'],
            'team': scorer['statistics'][0]['team']['name'],
            'goals': scorer['statistics'][0]['goals']['total'] or 0,
            'assists': scorer['statistics'][0]['goals']['assists'] or 0,
            'penalties': scorer['statistics'][0]['penalty']['scored'] or 0,
            'appearances': scorer['statistics'][0]['games']['appearences'] or 0,
            'minutes': scorer['statistics'][0]['games']['minutes'] or 0
        } for scorer in scorers])
        
        df['goals_per_90'] = round((df['goals'] / df['minutes']) * 90, 2)
        
        return df
    return None

# =============================================================================
# GET TOP ASSISTS
# =============================================================================

def get_top_assists(limit=20):
    """Get Premier League top assist providers"""
    data = make_request("players/topassists", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        assisters = data['response'][:limit]
        
        df = pd.DataFrame([{
            'player': player['player']['name'],
            'team': player['statistics'][0]['team']['name'],
            'assists': player['statistics'][0]['goals']['assists'] or 0,
            'goals': player['statistics'][0]['goals']['total'] or 0,
            'appearances': player['statistics'][0]['games']['appearences'] or 0,
            'minutes': player['statistics'][0]['games']['minutes'] or 0
        } for player in assisters])
        
        return df
    return None

# =============================================================================
# GET FIXTURES
# =============================================================================

def get_fixtures(status=None, from_date=None, to_date=None, round_num=None):
    """
    Get Premier League fixtures
    
    Parameters:
        status: NS (not started), FT (finished), LIVE, etc.
        from_date: Start date (YYYY-MM-DD)
        to_date: End date (YYYY-MM-DD)
        round_num: Specific matchday (e.g., "Regular Season - 1")
    """
    params = {
        "league": LEAGUE_ID,
        "season": SEASON
    }
    
    if status:
        params["status"] = status
    if from_date:
        params["from"] = from_date
    if to_date:
        params["to"] = to_date
    if round_num:
        params["round"] = f"Regular Season - {round_num}"
    
    data = make_request("fixtures", params=params)
    
    if data and data['response']:
        fixtures = data['response']
        
        df = pd.DataFrame([{
            'date': match['fixture']['date'][:10],
            'time': match['fixture']['date'][11:16],
            'home_team': match['teams']['home']['name'],
            'away_team': match['teams']['away']['name'],
            'home_score': match['goals']['home'],
            'away_score': match['goals']['away'],
            'status': match['fixture']['status']['short'],
            'round': match['league']['round'],
            'venue': match['fixture']['venue']['name'] if match['fixture']['venue'] else 'N/A'
        } for match in fixtures])
        
        # Sort by date
        df = df.sort_values('date').reset_index(drop=True)
        
        return df
    return None

# =============================================================================
# GET ALL TEAMS
# =============================================================================

def get_teams():
    """Get all Premier League teams"""
    data = make_request("teams", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        teams = data['response']
        
        df = pd.DataFrame([{
            'team_id': team['team']['id'],
            'name': team['team']['name'],
            'code': team['team']['code'],
            'founded': team['team']['founded'],
            'stadium': team['venue']['name'],
            'capacity': team['venue']['capacity'],
            'city': team['venue']['city']
        } for team in teams])
        
        return df
    return None

# =============================================================================
# GET PLAYER STATISTICS BY TEAM
# =============================================================================

def get_team_players(team_id):
    """Get all player statistics for a specific team"""
    data = make_request("players", params={
        "league": LEAGUE_ID,
        "season": SEASON,
        "team": team_id
    })
    
    if data and data['response']:
        players = data['response']
        
        df = pd.DataFrame([{
            'player': p['player']['name'],
            'age': p['player']['age'],
            'position': p['statistics'][0]['games']['position'],
            'appearances': p['statistics'][0]['games']['appearences'] or 0,
            'minutes': p['statistics'][0]['games']['minutes'] or 0,
            'goals': p['statistics'][0]['goals']['total'] or 0,
            'assists': p['statistics'][0]['goals']['assists'] or 0,
            'yellow_cards': p['statistics'][0]['cards']['yellow'] or 0,
            'red_cards': p['statistics'][0]['cards']['red'] or 0,
            'rating': p['statistics'][0]['games']['rating']
        } for p in players])
        
        return df
    return None

# =============================================================================
# GET TEAM STATISTICS
# =============================================================================

def get_team_stats(team_id):
    """Get detailed statistics for a team"""
    data = make_request("teams/statistics", params={
        "league": LEAGUE_ID,
        "season": SEASON,
        "team": team_id
    })
    
    if data and data['response']:
        stats = data['response']
        
        return {
            'team': stats['team']['name'],
            'form': stats['form'],
            'matches_played': stats['fixtures']['played']['total'],
            'wins': stats['fixtures']['wins']['total'],
            'draws': stats['fixtures']['draws']['total'],
            'losses': stats['fixtures']['loses']['total'],
            'goals_for': stats['goals']['for']['total']['total'],
            'goals_against': stats['goals']['against']['total']['total'],
            'clean_sheets': stats['clean_sheet']['total'],
            'penalty_scored': stats['penalty']['scored']['total'],
            'penalty_missed': stats['penalty']['missed']['total']
        }
    return None

# =============================================================================
# CHECK API STATUS / REMAINING REQUESTS
# =============================================================================

def check_api_status():
    """Check your API subscription status and remaining requests"""
    data = make_request("status")
    
    if data and data['response']:
        status = data['response']
        return {
            'account': status['account']['firstname'],
            'email': status['account']['email'],
            'plan': status['subscription']['plan'],
            'requests_today': status['requests']['current'],
            'requests_limit': status['requests']['limit_day']
        }
    return None

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    
    # Check API status first
    print("=" * 60)
    print("API STATUS")
    print("=" * 60)
    status = check_api_status()
    if status:
        for key, value in status.items():
            print(f"{key}: {value}")
    
    print("\n")
    
    # Get standings
    print("=" * 60)
    print("PREMIER LEAGUE STANDINGS 2024/25")
    print("=" * 60)
    standings = get_standings()
    if standings is not None:
        print(standings.to_string(index=False))
    
    print("\n")
    
    # Get top scorers
    print("=" * 60)
    print("TOP SCORERS")
    print("=" * 60)
    scorers = get_top_scorers(limit=10)
    if scorers is not None:
        print(scorers.to_string(index=False))
    
    print("\n")
    
    # Get top assists
    print("=" * 60)
    print("TOP ASSISTS")
    print("=" * 60)
    assists = get_top_assists(limit=10)
    if assists is not None:
        print(assists.to_string(index=False))
    
    print("\n")
    
    # Get recent results
    print("=" * 60)
    print("RECENT RESULTS (LAST 10)")
    print("=" * 60)
    results = get_fixtures(status="FT")
    if results is not None:
        print(results.tail(10).to_string(index=False))
    
    print("\n")
    
    # Get upcoming fixtures
    print("=" * 60)
    print("UPCOMING FIXTURES (NEXT 10)")
    print("=" * 60)
    upcoming = get_fixtures(status="NS")
    if upcoming is not None:
        print(upcoming.head(10).to_string(index=False))
    
    print("\n")
    
    # Get all teams
    print("=" * 60)
    print("ALL TEAMS")
    print("=" * 60)
    teams = get_teams()
    if teams is not None:
        print(teams.to_string(index=False))

API STATUS
account: Alfie
email: alfie.yearsley@leonardcurtis.co.uk
plan: Free
requests_today: 0
requests_limit: 100


PREMIER LEAGUE STANDINGS 2024/25
 position              team  played  won  drawn  lost  goals_for  goals_against  goal_difference  points  form
        1         Liverpool      38   25      9     4         86             41               45      84 DLDLW
        2           Arsenal      38   20     14     4         69             34               35      74 WWDLD
        3   Manchester City      38   21      8     9         72             44               28      71 WWDWW
        4           Chelsea      38   20      9     9         64             43               21      69 WWLWW
        5         Newcastle      38   20      6    12         68             47               21      66 LLWDW
        6       Aston Villa      38   19      9    10         58             51                7      66 LWWWL
        7 Nottingham Forest      38   19      8    11         58       

###

### Code to obtain; Standings, Teams, Top Goal Scorers, Highest Number Of Assists and Upcoming Fixtures

###

In [2]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# =============================================================================
# PANDAS DISPLAY SETTINGS
# =============================================================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# =============================================================================
# CONFIGURATION
# =============================================================================

API_KEY = "your_api_key_here"  # Replace with your API-Football key
BASE_URL = "https://v3.football.api-sports.io"

LEAGUE_ID = 39
SEASON = 2024

headers = {
    "x-apisports-key": API_KEY
}

# =============================================================================
# HELPER FUNCTION
# =============================================================================

def make_request(endpoint, params=None):
    """Make API request and return JSON data"""
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data.get('errors') and len(data['errors']) > 0:
            print(f"API Error: {data['errors']}")
            return None
        return data
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# =============================================================================
# GET LEAGUE STANDINGS
# =============================================================================

def get_standings():
    """Get current Premier League table"""
    data = make_request("standings", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        standings = data['response'][0]['league']['standings'][0]
        
        df = pd.DataFrame([{
            'position': team['rank'],
            'team': team['team']['name'],
            'played': team['all']['played'],
            'won': team['all']['win'],
            'drawn': team['all']['draw'],
            'lost': team['all']['lose'],
            'goals_for': team['all']['goals']['for'],
            'goals_against': team['all']['goals']['against'],
            'goal_difference': team['goalsDiff'],
            'points': team['points'],
            'form': team['form']
        } for team in standings])
        
        return df
    return None

# =============================================================================
# GET TOP SCORERS
# =============================================================================

def get_top_scorers(limit=20):
    """Get Premier League top scorers"""
    print("Fetching top scorers...")
    
    data = make_request("players/topscorers", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        scorers = data['response'][:limit]
        
        df = pd.DataFrame([{
            'rank': idx + 1,
            'player': scorer['player']['name'],
            'team': scorer['statistics'][0]['team']['name'],
            'goals': scorer['statistics'][0]['goals']['total'] or 0,
            'assists': scorer['statistics'][0]['goals']['assists'] or 0,
            'penalties': scorer['statistics'][0]['penalty']['scored'] or 0,
            'appearances': scorer['statistics'][0]['games']['appearences'] or 0,
            'minutes': scorer['statistics'][0]['games']['minutes'] or 0
        } for idx, scorer in enumerate(scorers)])
        
        df['goals_per_90'] = round((df['goals'] / df['minutes'].replace(0, 1)) * 90, 2)
        
        return df
    return None

# =============================================================================
# GET TOP ASSISTS
# =============================================================================

def get_top_assists(limit=20):
    """Get Premier League top assist providers"""
    print("Fetching top assists...")
    
    data = make_request("players/topassists", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        assisters = data['response'][:limit]
        
        df = pd.DataFrame([{
            'rank': idx + 1,
            'player': player['player']['name'],
            'team': player['statistics'][0]['team']['name'],
            'assists': player['statistics'][0]['goals']['assists'] or 0,
            'goals': player['statistics'][0]['goals']['total'] or 0,
            'appearances': player['statistics'][0]['games']['appearences'] or 0,
            'minutes': player['statistics'][0]['games']['minutes'] or 0
        } for idx, player in enumerate(assisters)])
        
        return df
    return None

# =============================================================================
# GET FIXTURES
# =============================================================================

def get_fixtures(status=None, limit=None):
    """Get Premier League fixtures"""
    print(f"Fetching fixtures (status: {status})...")
    
    params = {
        "league": LEAGUE_ID,
        "season": SEASON
    }
    
    if status:
        params["status"] = status
    
    data = make_request("fixtures", params=params)
    
    if data and data['response']:
        fixtures = data['response']
        
        df = pd.DataFrame([{
            'date': match['fixture']['date'][:10],
            'time': match['fixture']['date'][11:16],
            'home_team': match['teams']['home']['name'],
            'away_team': match['teams']['away']['name'],
            'home_score': match['goals']['home'],
            'away_score': match['goals']['away'],
            'status': match['fixture']['status']['short'],
            'round': match['league']['round'],
            'venue': match['fixture']['venue']['name'] if match['fixture']['venue'] else 'N/A'
        } for match in fixtures])
        
        df = df.sort_values('date').reset_index(drop=True)
        
        if limit:
            return df.head(limit)
        return df
    return None

# =============================================================================
# GET ALL TEAMS
# =============================================================================

def get_teams():
    """Get all Premier League teams"""
    print("Fetching teams...")
    
    data = make_request("teams", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        teams = data['response']
        
        df = pd.DataFrame([{
            'team_id': team['team']['id'],
            'name': team['team']['name'],
            'code': team['team']['code'],
            'founded': team['team']['founded'],
            'stadium': team['venue']['name'],
            'capacity': team['venue']['capacity'],
            'city': team['venue']['city']
        } for team in teams])
        
        return df
    return None

# =============================================================================
# CHECK API STATUS
# =============================================================================

def check_api_status():
    """Check your API subscription status and remaining requests"""
    data = make_request("status")
    
    if data and data['response']:
        status = data['response']
        return {
            'account': status['account']['firstname'],
            'email': status['account']['email'],
            'plan': status['subscription']['plan'],
            'requests_today': status['requests']['current'],
            'requests_limit': status['requests']['limit_day']
        }
    return None

# =============================================================================
# EXPORT TO CSV
# =============================================================================

def save_to_csv(df, filename):
    """Save DataFrame to CSV with error handling"""
    if df is None:
        print(f"  ✗ No data for {filename}")
        return False
    
    try:
        df.to_csv(filename, index=False)
        print(f"  ✓ Saved {filename}: {len(df)} rows")
        return True
    except PermissionError:
        backup = f"{filename.replace('.csv', '')}_{datetime.now().strftime('%H%M%S')}.csv"
        df.to_csv(backup, index=False)
        print(f"  ⚠️ {filename} was open - saved to {backup} instead")
        return True
    except Exception as e:
        print(f"  ✗ Error saving {filename}: {e}")
        return False

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    
    # Check API status first
    print("=" * 60)
    print("API STATUS")
    print("=" * 60)
    status = check_api_status()
    if status:
        for key, value in status.items():
            print(f"{key}: {value}")
    
    print("\n")
    print("=" * 60)
    print("FETCHING DATA FOR POWER BI")
    print("=" * 60)
    
    # Collect all data
    standings = get_standings()
    scorers = get_top_scorers(limit=20)
    assists = get_top_assists(limit=20)
    results = get_fixtures(status="FT")
    upcoming = get_fixtures(status="NS")
    teams = get_teams()
    
    print("\n")
    print("=" * 60)
    print("EXPORTING TO CSV")
    print("=" * 60)
    
    # Save each dataset to CSV
    save_to_csv(standings, "pl_standings.csv")
    save_to_csv(scorers, "pl_top_scorers.csv")
    save_to_csv(assists, "pl_top_assists.csv")
    save_to_csv(results, "pl_results.csv")
    save_to_csv(upcoming, "pl_upcoming_fixtures.csv")
    save_to_csv(teams, "pl_teams.csv")
    
    print("\n")
    print("=" * 60)
    print("DONE - Ready for Power BI")
    print("=" * 60)
    print("\nFiles created:")
    print("  - pl_standings.csv")
    print("  - pl_top_scorers.csv")
    print("  - pl_top_assists.csv")
    print("  - pl_results.csv")
    print("  - pl_upcoming_fixtures.csv")
    print("  - pl_teams.csv")
    print("\nTo import into Power BI:")
    print("1. Open Power BI Desktop")
    print("2. Click 'Get Data' > 'Text/CSV'")
    print("3. Select each CSV file")
    print("4. Click 'Load'")
    print("5. Create relationships between tables using 'team' columns")

API STATUS
account: Alfie
email: alfie.yearsley@leonardcurtis.co.uk
plan: Free
requests_today: 68
requests_limit: 100


FETCHING DATA FOR POWER BI
Fetching top scorers...
Fetching top assists...
Fetching fixtures (status: FT)...
Fetching fixtures (status: NS)...
Fetching teams...


EXPORTING TO CSV
  ✓ Saved pl_standings.csv: 20 rows
  ✓ Saved pl_top_scorers.csv: 19 rows
  ✓ Saved pl_top_assists.csv: 19 rows
  ✓ Saved pl_results.csv: 380 rows
  ✗ No data for pl_upcoming_fixtures.csv
  ✓ Saved pl_teams.csv: 20 rows


DONE - Ready for Power BI

Files created:
  - pl_standings.csv
  - pl_top_scorers.csv
  - pl_top_assists.csv
  - pl_results.csv
  - pl_upcoming_fixtures.csv
  - pl_teams.csv

To import into Power BI:
1. Open Power BI Desktop
2. Click 'Get Data' > 'Text/CSV'
3. Select each CSV file
4. Click 'Load'
5. Create relationships between tables using 'team' columns


###

### First Attempt of obtaining top player ratings

###

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# =============================================================================
# PANDAS DISPLAY SETTINGS
# =============================================================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# =============================================================================
# CONFIGURATION
# =============================================================================

API_KEY = "your_api_key_here"  # Replace with your API-Football key
BASE_URL = "https://v3.football.api-sports.io"

LEAGUE_ID = 39
SEASON = 2024

headers = {
    "x-apisports-key": API_KEY
}

# =============================================================================
# HELPER FUNCTION
# =============================================================================

def make_request(endpoint, params=None):
    """Make API request and return JSON data"""
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data.get('errors') and len(data['errors']) > 0:
            print(f"API Error: {data['errors']}")
            return None
        return data
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# =============================================================================
# GET LEAGUE STANDINGS
# =============================================================================

def get_standings():
    """Get current Premier League table"""
    data = make_request("standings", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        standings = data['response'][0]['league']['standings'][0]
        
        df = pd.DataFrame([{
            'position': team['rank'],
            'team': team['team']['name'],
            'played': team['all']['played'],
            'won': team['all']['win'],
            'drawn': team['all']['draw'],
            'lost': team['all']['lose'],
            'goals_for': team['all']['goals']['for'],
            'goals_against': team['all']['goals']['against'],
            'goal_difference': team['goalsDiff'],
            'points': team['points'],
            'form': team['form']
        } for team in standings])
        
        return df
    return None

# =============================================================================
# GET TOP SCORERS
# =============================================================================

def get_top_scorers(limit=20):
    """Get Premier League top scorers"""
    print("Fetching top scorers...")
    
    data = make_request("players/topscorers", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        scorers = data['response'][:limit]
        
        df = pd.DataFrame([{
            'rank': idx + 1,
            'player': scorer['player']['name'],
            'team': scorer['statistics'][0]['team']['name'],
            'goals': scorer['statistics'][0]['goals']['total'] or 0,
            'assists': scorer['statistics'][0]['goals']['assists'] or 0,
            'penalties': scorer['statistics'][0]['penalty']['scored'] or 0,
            'appearances': scorer['statistics'][0]['games']['appearences'] or 0,
            'minutes': scorer['statistics'][0]['games']['minutes'] or 0
        } for idx, scorer in enumerate(scorers)])
        
        df['goals_per_90'] = round((df['goals'] / df['minutes'].replace(0, 1)) * 90, 2)
        
        return df
    return None

# =============================================================================
# GET TOP ASSISTS
# =============================================================================

def get_top_assists(limit=20):
    """Get Premier League top assist providers"""
    print("Fetching top assists...")
    
    data = make_request("players/topassists", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        assisters = data['response'][:limit]
        
        df = pd.DataFrame([{
            'rank': idx + 1,
            'player': player['player']['name'],
            'team': player['statistics'][0]['team']['name'],
            'assists': player['statistics'][0]['goals']['assists'] or 0,
            'goals': player['statistics'][0]['goals']['total'] or 0,
            'appearances': player['statistics'][0]['games']['appearences'] or 0,
            'minutes': player['statistics'][0]['games']['minutes'] or 0
        } for idx, player in enumerate(assisters)])
        
        return df
    return None

# =============================================================================
# GET FIXTURES
# =============================================================================

def get_fixtures(status=None, limit=200):
    """
    Get Premier League fixtures
    
    Parameters:
        status: NS (not started), FT (finished), LIVE, etc.
        limit: Max results to return
    """
    print("Fetching fixtures...")
    
    params = {
        "league": LEAGUE_ID,
        "season": SEASON
    }
    
    if status:
        params["status"] = status
    
    data = make_request("fixtures", params=params)
    
    if data and data['response']:
        fixtures = data['response']
        
        df = pd.DataFrame([{
            'date': match['fixture']['date'][:10],
            'time': match['fixture']['date'][11:16],
            'home_team': match['teams']['home']['name'],
            'away_team': match['teams']['away']['name'],
            'home_score': match['goals']['home'],
            'away_score': match['goals']['away'],
            'status': match['fixture']['status']['short'],
            'round': match['league']['round'],
            'venue': match['fixture']['venue']['name'] if match['fixture']['venue'] else 'N/A'
        } for match in fixtures])
        
        df = df.sort_values('date').reset_index(drop=True)
        
        return df.head(limit)
    return None

# =============================================================================
# GET ALL TEAMS
# =============================================================================

def get_teams():
    """Get all Premier League teams"""
    data = make_request("teams", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if data and data['response']:
        teams = data['response']
        
        df = pd.DataFrame([{
            'team_id': team['team']['id'],
            'name': team['team']['name'],
            'code': team['team']['code'],
            'founded': team['team']['founded'],
            'stadium': team['venue']['name'],
            'capacity': team['venue']['capacity'],
            'city': team['venue']['city']
        } for team in teams])
        
        return df
    return None

# =============================================================================
# GET TOP PLAYERS BY RATING
# =============================================================================

def get_top_players_by_rating(limit=50, min_appearances=5):
    """
    Get top players ordered by rating
    
    Parameters:
        limit: Number of players to return
        min_appearances: Minimum appearances to qualify
    """
    print("Fetching top players by rating (fetching by team)...")
    
    # First get all teams
    teams_data = make_request("teams", params={
        "league": LEAGUE_ID,
        "season": SEASON
    })
    
    if not teams_data or not teams_data['response']:
        print("Could not fetch teams")
        return None
    
    all_players = []
    
    for team in teams_data['response']:
        team_id = team['team']['id']
        team_name = team['team']['name']
        print(f"  Fetching {team_name}...")
        
        # Get players for this team (first page only to save API calls)
        data = make_request("players", params={
            "league": LEAGUE_ID,
            "season": SEASON,
            "team": team_id
        })
        
        if data and data['response']:
            all_players.extend(data['response'])
    
    print(f"  Total players fetched: {len(all_players)}")
    
    if all_players:
        df = pd.DataFrame([{
            'player': p['player']['name'],
            'team': p['statistics'][0]['team']['name'],
            'age': p['player']['age'],
            'nationality': p['player']['nationality'],
            'position': p['statistics'][0]['games']['position'],
            'appearances': p['statistics'][0]['games']['appearences'] or 0,
            'starts': p['statistics'][0]['games']['lineups'] or 0,
            'minutes': p['statistics'][0]['games']['minutes'] or 0,
            'goals': p['statistics'][0]['goals']['total'] or 0,
            'assists': p['statistics'][0]['goals']['assists'] or 0,
            'yellow_cards': p['statistics'][0]['cards']['yellow'] or 0,
            'red_cards': p['statistics'][0]['cards']['red'] or 0,
            'rating': float(p['statistics'][0]['games']['rating']) if p['statistics'][0]['games']['rating'] else 0
        } for p in all_players])
        
        # Filter players with minimum appearances
        df = df[df['appearances'] >= min_appearances]
        
        # Sort by rating descending
        df = df.sort_values('rating', ascending=False).reset_index(drop=True)
        
        # Add rank
        df.insert(0, 'rank', range(1, len(df) + 1))
        
        return df.head(limit)
    return None

# =============================================================================
# CHECK API STATUS
# =============================================================================

def check_api_status():
    """Check your API subscription status and remaining requests"""
    data = make_request("status")
    
    if data and data['response']:
        status = data['response']
        return {
            'account': status['account']['firstname'],
            'email': status['account']['email'],
            'plan': status['subscription']['plan'],
            'requests_today': status['requests']['current'],
            'requests_limit': status['requests']['limit_day']
        }
    return None

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    
    # Check API status first
    print("=" * 60)
    print("API STATUS")
    print("=" * 60)
    status = check_api_status()
    if status:
        for key, value in status.items():
            print(f"{key}: {value}")
    
    print("\n")
    
    # Get standings (1 API call)
    print("=" * 60)
    print("PREMIER LEAGUE STANDINGS 2024/25")
    print("=" * 60)
    standings = get_standings()
    if standings is not None:
        print(standings.to_string(index=False))
    
    print("\n")
    
    # Get top scorers (1 API call)
    print("=" * 60)
    print("TOP 20 SCORERS")
    print("=" * 60)
    scorers = get_top_scorers(limit=20)
    if scorers is not None:
        print(scorers.to_string(index=False))
    
    print("\n")
    
    # Get top assists (1 API call)
    print("=" * 60)
    print("TOP 20 ASSISTS")
    print("=" * 60)
    assists = get_top_assists(limit=20)
    if assists is not None:
        print(assists.to_string(index=False))
    
    print("\n")
    
    # Get all teams (1 API call)
    print("=" * 60)
    print("ALL TEAMS")
    print("=" * 60)
    teams = get_teams()
    if teams is not None:
        print(teams.to_string(index=False))
    
    print("\n")
    
    # Get top 50 players by rating (21 API calls - 1 for teams + 20 for each team)
    print("=" * 60)
    print("TOP 50 PLAYERS BY RATING")
    print("=" * 60)
    top_players = get_top_players_by_rating(limit=50, min_appearances=5)
    if top_players is not None:
        print(top_players.to_string(index=False))

    print("\n")
    
    # Get results - limited to 200 (1 API call)
    print("=" * 60)
    print("ALL RESULTS THIS SEASON (MAX 200)")
    print("=" * 60)
    results = get_fixtures(status="FT", limit=200)
    if results is not None:
        print(f"Total matches shown: {len(results)}")
        print(results.to_string(index=False))

API STATUS
account: Alfie
email: alfie.yearsley@leonardcurtis.co.uk
plan: Free
requests_today: 60
requests_limit: 100


PREMIER LEAGUE STANDINGS 2024/25
 position              team  played  won  drawn  lost  goals_for  goals_against  goal_difference  points  form
        1         Liverpool      38   25      9     4         86             41               45      84 DLDLW
        2           Arsenal      38   20     14     4         69             34               35      74 WWDLD
        3   Manchester City      38   21      8     9         72             44               28      71 WWDWW
        4           Chelsea      38   20      9     9         64             43               21      69 WWLWW
        5         Newcastle      38   20      6    12         68             47               21      66 LLWDW
        6       Aston Villa      38   19      9    10         58             51                7      66 LWWWL
        7 Nottingham Forest      38   19      8    11         58      

###

### Wait 1 minute per team, to obtain highest player rankings. Wait was used to as restricted to 10 API calls per minute 

###

In [ ]:
import requests
import pandas as pd
import time

# =============================================================================
# PANDAS DISPLAY SETTINGS
# =============================================================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# =============================================================================
# CONFIGURATION
# =============================================================================

API_KEY = "your_api_key_here"  # Replace with your API-Football key
BASE_URL = "https://v3.football.api-sports.io"

LEAGUE_ID = 39
SEASON = 2024

headers = {
    "x-apisports-key": API_KEY
}

# All 20 Premier League teams alphabetically with their IDs
TEAMS_ALPHABETICAL = [
    (42, "Arsenal"),
    (66, "Aston Villa"),
    (35, "Bournemouth"),
    (55, "Brentford"),
    (51, "Brighton"),
    (49, "Chelsea"),
    (52, "Crystal Palace"),
    (45, "Everton"),
    (36, "Fulham"),
    (57, "Ipswich"),
    (46, "Leicester"),
    (40, "Liverpool"),
    (50, "Manchester City"),
    (33, "Manchester United"),
    (34, "Newcastle"),
    (65, "Nottingham Forest"),
    (41, "Southampton"),
    (47, "Tottenham"),
    (48, "West Ham"),
    (39, "Wolves")
]

BATCH_SIZE = 2
WAIT_TIME = 60  # seconds between batches

# =============================================================================
# HELPER FUNCTION
# =============================================================================

def make_request(endpoint, params=None):
    """Make API request and return JSON data"""
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data.get('errors') and len(data['errors']) > 0:
            print(f"API Error: {data['errors']}")
            return None
        return data
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# =============================================================================
# FETCH ALL PLAYERS IN BATCHES
# =============================================================================

def fetch_all_players_in_batches():
    """Fetch players from all 20 teams, 2 at a time with 60s pauses"""
    
    all_players = []
    total_batches = len(TEAMS_ALPHABETICAL) // BATCH_SIZE
    
    for batch_num in range(total_batches):
        start_idx = batch_num * BATCH_SIZE
        end_idx = start_idx + BATCH_SIZE
        batch_teams = TEAMS_ALPHABETICAL[start_idx:end_idx]
        
        print("=" * 60)
        print(f"BATCH {batch_num + 1}/{total_batches}")
        print("=" * 60)
        
        for team_id, team_name in batch_teams:
            print(f"  Fetching {team_name}...")
            
            data = make_request("players", params={
                "league": LEAGUE_ID,
                "season": SEASON,
                "team": team_id
            })
            
            if data and data['response']:
                print(f"    ✓ {len(data['response'])} players")
                all_players.extend(data['response'])
            else:
                print(f"    ✗ Failed to fetch")
        
        # Save progress after each batch
        save_progress(all_players, batch_num + 1)
        
        # Wait before next batch (unless it's the last one)
        if batch_num < total_batches - 1:
            print(f"\n  Waiting {WAIT_TIME} seconds before next batch...")
            print(f"  Progress: {len(all_players)} players collected so far\n")
            time.sleep(WAIT_TIME)
    
    return all_players

# =============================================================================
# SAVE PROGRESS
# =============================================================================

def save_progress(players, batch_num):
    """Save current progress to CSV"""
    
    if not players:
        return
    
    df = pd.DataFrame([{
        'player': p['player']['name'],
        'team': p['statistics'][0]['team']['name'],
        'position': p['statistics'][0]['games']['position'],
        'appearances': p['statistics'][0]['games']['appearences'] or 0,
        'minutes': p['statistics'][0]['games']['minutes'] or 0,
        'goals': p['statistics'][0]['goals']['total'] or 0,
        'assists': p['statistics'][0]['goals']['assists'] or 0,
        'rating': float(p['statistics'][0]['games']['rating']) if p['statistics'][0]['games']['rating'] else 0
    } for p in players])
    
    # Filter and sort
    df = df[df['appearances'] >= 3]
    df = df.sort_values('rating', ascending=False).reset_index(drop=True)
    df.insert(0, 'rank', range(1, len(df) + 1))
    
    filename = f"pl_players_batch_{batch_num}.csv"
    df.to_csv(filename, index=False)
    print(f"  Progress saved to {filename}")

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    
    print("=" * 60)
    print("PREMIER LEAGUE PLAYER RATINGS - FULL COLLECTION")
    print("=" * 60)
    print(f"Fetching {len(TEAMS_ALPHABETICAL)} teams in batches of {BATCH_SIZE}")
    print(f"Wait time between batches: {WAIT_TIME} seconds")
    print(f"Total estimated time: {(len(TEAMS_ALPHABETICAL) // BATCH_SIZE - 1) * WAIT_TIME // 60} minutes")
    print("=" * 60)
    print()
    
    all_players = fetch_all_players_in_batches()
    
    # Final save with all players
    print("\n")
    print("=" * 60)
    print("COMPLETE - FINAL RESULTS")
    print("=" * 60)
    
    if all_players:
        df = pd.DataFrame([{
            'player': p['player']['name'],
            'team': p['statistics'][0]['team']['name'],
            'position': p['statistics'][0]['games']['position'],
            'appearances': p['statistics'][0]['games']['appearences'] or 0,
            'minutes': p['statistics'][0]['games']['minutes'] or 0,
            'goals': p['statistics'][0]['goals']['total'] or 0,
            'assists': p['statistics'][0]['goals']['assists'] or 0,
            'rating': float(p['statistics'][0]['games']['rating']) if p['statistics'][0]['games']['rating'] else 0
        } for p in all_players])
        
        df = df[df['appearances'] >= 3]
        df = df.sort_values('rating', ascending=False).reset_index(drop=True)
        df.insert(0, 'rank', range(1, len(df) + 1))
        
        print(f"\nTotal players with 3+ appearances: {len(df)}")
        print("\nTOP 50 PLAYERS BY RATING:")
        print(df.head(50).to_string(index=False))
        
        df.to_csv("pl_players_FINAL.csv", index=False)
        print("\nFinal results saved to pl_players_FINAL.csv")

PREMIER LEAGUE PLAYER RATINGS - FULL COLLECTION
Fetching 20 teams in batches of 2
Wait time between batches: 60 seconds
Total estimated time: 9 minutes

BATCH 1/10
  Fetching Arsenal...
    ✓ 20 players
  Fetching Aston Villa...
    ✓ 20 players
  Progress saved to pl_players_batch_1.csv

  Waiting 60 seconds before next batch...
  Progress: 40 players collected so far

BATCH 2/10
  Fetching Bournemouth...
    ✓ 20 players
  Fetching Brentford...
    ✓ 20 players
  Progress saved to pl_players_batch_2.csv

  Waiting 60 seconds before next batch...
  Progress: 80 players collected so far

BATCH 3/10
  Fetching Brighton...
    ✓ 20 players
  Fetching Chelsea...
    ✓ 20 players
  Progress saved to pl_players_batch_3.csv

  Waiting 60 seconds before next batch...
  Progress: 120 players collected so far

BATCH 4/10
  Fetching Crystal Palace...
    ✓ 20 players
  Fetching Everton...
    ✓ 20 players
  Progress saved to pl_players_batch_4.csv

  Waiting 60 seconds before next batch...
  Pro

###

### Exporting Data in Excel Files

###

In [ ]:
import requests
import pandas as pd
import time

# =============================================================================
# PANDAS DISPLAY SETTINGS
# =============================================================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# =============================================================================
# CONFIGURATION
# =============================================================================

API_KEY = "your_api_key_here"  # Replace with your API-Football key
BASE_URL = "https://v3.football.api-sports.io"

LEAGUE_ID = 39
SEASON = 2024

headers = {
    "x-apisports-key": API_KEY
}

# All 20 Premier League teams alphabetically with their IDs
TEAMS_ALPHABETICAL = [
    (42, "Arsenal"),
    (66, "Aston Villa"),
    (35, "Bournemouth"),
    (55, "Brentford"),
    (51, "Brighton"),
    (49, "Chelsea"),
    (52, "Crystal Palace"),
    (45, "Everton"),
    (36, "Fulham"),
    (57, "Ipswich"),
    (46, "Leicester"),
    (40, "Liverpool"),
    (50, "Manchester City"),
    (33, "Manchester United"),
    (34, "Newcastle"),
    (65, "Nottingham Forest"),
    (41, "Southampton"),
    (47, "Tottenham"),
    (48, "West Ham"),
    (39, "Wolves")
]

BATCH_SIZE = 2
WAIT_TIME = 60  # seconds between batches

# =============================================================================
# HELPER FUNCTION
# =============================================================================

def make_request(endpoint, params=None):
    """Make API request and return JSON data"""
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data.get('errors') and len(data['errors']) > 0:
            print(f"API Error: {data['errors']}")
            return None
        return data
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# =============================================================================
# FETCH ALL PLAYERS IN BATCHES
# =============================================================================

def fetch_all_players_in_batches():
    """Fetch players from all 20 teams, 2 at a time with 60s pauses"""
    
    all_players = []
    total_batches = len(TEAMS_ALPHABETICAL) // BATCH_SIZE
    
    for batch_num in range(total_batches):
        start_idx = batch_num * BATCH_SIZE
        end_idx = start_idx + BATCH_SIZE
        batch_teams = TEAMS_ALPHABETICAL[start_idx:end_idx]
        
        print("=" * 60)
        print(f"BATCH {batch_num + 1}/{total_batches}")
        print("=" * 60)
        
        for team_id, team_name in batch_teams:
            print(f"  Fetching {team_name}...")
            
            data = make_request("players", params={
                "league": LEAGUE_ID,
                "season": SEASON,
                "team": team_id
            })
            
            if data and data['response']:
                print(f"    ✓ {len(data['response'])} players")
                all_players.extend(data['response'])
            else:
                print(f"    ✗ Failed to fetch")
        
        # Save progress after each batch
        save_progress(all_players, batch_num + 1)
        
        # Wait before next batch (unless it's the last one)
        if batch_num < total_batches - 1:
            print(f"\n  Waiting {WAIT_TIME} seconds before next batch...")
            print(f"  Progress: {len(all_players)} players collected so far\n")
            time.sleep(WAIT_TIME)
    
    return all_players

# =============================================================================
# SAVE PROGRESS
# =============================================================================

def save_progress(players, batch_num):
    """Save current progress to CSV"""
    
    if not players:
        return
    
    df = pd.DataFrame([{
        'player': p['player']['name'],
        'team': p['statistics'][0]['team']['name'],
        'position': p['statistics'][0]['games']['position'],
        'appearances': p['statistics'][0]['games']['appearences'] or 0,
        'minutes': p['statistics'][0]['games']['minutes'] or 0,
        'goals': p['statistics'][0]['goals']['total'] or 0,
        'assists': p['statistics'][0]['goals']['assists'] or 0,
        'rating': float(p['statistics'][0]['games']['rating']) if p['statistics'][0]['games']['rating'] else 0
    } for p in players])
    
    # Filter: minimum 3 appearances, 90+ minutes, and must have a rating
    df = df[(df['appearances'] >= 3) & (df['minutes'] >= 90) & (df['rating'] > 0)]
    df = df.sort_values('rating', ascending=False).reset_index(drop=True)
    df.insert(0, 'rank', range(1, len(df) + 1))
    
    filename = f"pl_players_batch_{batch_num}.csv"
    df.to_csv(filename, index=False)
    print(f"  Progress saved to {filename}")

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    
    print("=" * 60)
    print("PREMIER LEAGUE PLAYER RATINGS - FULL COLLECTION")
    print("=" * 60)
    print(f"Fetching {len(TEAMS_ALPHABETICAL)} teams in batches of {BATCH_SIZE}")
    print(f"Wait time between batches: {WAIT_TIME} seconds")
    print(f"Total estimated time: {(len(TEAMS_ALPHABETICAL) // BATCH_SIZE - 1) * WAIT_TIME // 60} minutes")
    print("=" * 60)
    print()
    
    all_players = fetch_all_players_in_batches()
    
    # Final save with all players
    print("\n")
    print("=" * 60)
    print("COMPLETE - FINAL RESULTS")
    print("=" * 60)
    
    if all_players:
        df = pd.DataFrame([{
            'player': p['player']['name'],
            'team': p['statistics'][0]['team']['name'],
            'position': p['statistics'][0]['games']['position'],
            'appearances': p['statistics'][0]['games']['appearences'] or 0,
            'minutes': p['statistics'][0]['games']['minutes'] or 0,
            'goals': p['statistics'][0]['goals']['total'] or 0,
            'assists': p['statistics'][0]['goals']['assists'] or 0,
            'rating': float(p['statistics'][0]['games']['rating']) if p['statistics'][0]['games']['rating'] else 0
        } for p in all_players])
        
        # Filter: minimum 3 appearances, 90+ minutes, and must have a rating
        df = df[(df['appearances'] >= 3) & (df['minutes'] >= 90) & (df['rating'] > 0)]
        df = df.sort_values('rating', ascending=False).reset_index(drop=True)
        df.insert(0, 'rank', range(1, len(df) + 1))
        
        print(f"\nTotal players with valid ratings: {len(df)}")
        print("\nTOP 50 PLAYERS BY RATING:")
        print(df.head(50).to_string(index=False))
        
        df.to_csv("pl_players_FINAL.csv", index=False)
        print("\nFinal results saved to pl_players_FINAL.csv")

PREMIER LEAGUE PLAYER RATINGS - FULL COLLECTION
Fetching 20 teams in batches of 2
Wait time between batches: 60 seconds
Total estimated time: 9 minutes

BATCH 1/10
  Fetching Arsenal...
    ✓ 20 players
  Fetching Aston Villa...
    ✓ 20 players
  Progress saved to pl_players_batch_1.csv

  Waiting 60 seconds before next batch...
  Progress: 40 players collected so far

BATCH 2/10
  Fetching Bournemouth...
    ✓ 20 players
  Fetching Brentford...
    ✓ 20 players
  Progress saved to pl_players_batch_2.csv

  Waiting 60 seconds before next batch...
  Progress: 80 players collected so far

BATCH 3/10
  Fetching Brighton...
    ✓ 20 players
  Fetching Chelsea...
    ✓ 20 players
  Progress saved to pl_players_batch_3.csv

  Waiting 60 seconds before next batch...
  Progress: 120 players collected so far

BATCH 4/10
  Fetching Crystal Palace...
    ✓ 20 players
  Fetching Everton...
    ✓ 20 players
  Progress saved to pl_players_batch_4.csv

  Waiting 60 seconds before next batch...
  Pro